instal

In [ ]:
!pip install pandas numpy scikit-learn nltk matplotlib seaborn

In [ ]:
import re
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split

nltk.download('stopwords')
nltk.download('punkt')

Load Seluruh 50.000 Data dan Preprocessing

In [ ]:
# Membaca seluruh data (50.000 baris)
df = pd.read_csv('IMDB Dataset.csv')
print(f'Total data yang dimuat: {len(df):,} baris')

# Stopwords bahasa Inggris
stop_words = set(stopwords.words('english'))


def clean_text(text):
  text = re.sub(r'<.*?>', ' ', text)  # Hapus tag HTML
  text = re.sub(r'[^a-zA-Z]', ' ', text)  # Hapus karakter selain huruf
  text = text.lower()  # Lowercase
  tokens = [
      word
      for word in text.split()
      if word not in stop_words and len(word) > 2
  ]
  return ' '.join(tokens)


# Membersihkan seluruh 50.000 ulasan
df['cleaned_review'] = df['review'].apply(clean_text)
print('Preprocessing selesai untuk seluruh 50.000 ulasan.')

Ekstraksi Fitur Seluruh Dokumen & Tabel Frekuensi Term

In [ ]:
# Ekstraksi seluruh term di 50.000 dokumen
count_vec = CountVectorizer()
dtm_sparse = count_vec.fit_transform(df['cleaned_review'])

terms = count_vec.get_feature_names_out()

# Menghitung total kemunculan setiap term pada seluruh 50.000 baris
term_counts = np.asarray(dtm_sparse.sum(axis=0)).flatten()

# Membuat tabel: Kolom = Semua Term di CSV, Baris = Total kemunculan di 50k data
df_term_frequency = pd.DataFrame([term_counts], columns=terms)

print(f'Total Vocabulary Unik: {len(terms):,} kata')
print(f'Dimensi Tabel: {df_term_frequency.shape}')

# Menampilkan 10 term teratas dengan frekuensi tertinggi
top_10_terms = (
    df_term_frequency.iloc[0].sort_values(ascending=False).head(10).to_frame().T
)
print('\n10 Term yang Paling Sering Muncul di Seluruh CSV:')
display(top_10_terms)

Clustering K-Means ($k=2$) pada Seluruh 50.000 Dokumen

In [ ]:
# 1. Transformasi TF-IDF untuk seluruh 50.000 dokumen
# max_features=10000 digunakan agar representasi fitur tetap padat dan komputasi efisien
tfidf_cluster = TfidfVectorizer(max_features=10000, min_df=5, max_df=0.8)
tfidf_matrix_50k = tfidf_cluster.fit_transform(df['cleaned_review'])
vocab_cluster = tfidf_cluster.get_feature_names_out()

print(f'Dimensi Matriks Input Clustering: {tfidf_matrix_50k.shape}')
# Output: (50000, 10000) -> 50.000 Dokumen x 10.000 Vocabulary

# 2. Training K-Means dengan k=2 pada seluruh 50.000 dokumen
kmeans = KMeans(n_clusters=2, random_state=42, n_init=5)
df['cluster'] = kmeans.fit_predict(tfidf_matrix_50k)

# 3. Interpretasi Karakteristik Cluster 0 dan Cluster 1
print('\n' + '=' * 50)
print('INTERPRETASI CLUSTER (DARI 50.000 DOKUMEN)')
print('=' * 50)

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
for i in range(2):
  top_terms_cluster = [vocab_cluster[ind] for ind in order_centroids[i, :10]]
  count_cluster = (df['cluster'] == i).sum()
  print(f'\n--- Cluster {i} (Total: {count_cluster:,} dokumen) ---')
  print(f"Top 10 Kata Dominan: {', '.join(top_terms_cluster)}")

  # Cross-tabulation terhadap sentimen aktual untuk melihat pola pemisahan
  sentiment_dist = df[df['cluster'] == i]['sentiment'].value_counts().to_dict()
  print(f'Distribusi Sentimen Aktual: {sentiment_dist}')

# 4. Evaluasi Clustering
# Evaluasi menggunakan sample 10.000 data agar komputasi Silhouette Score efisien dan tidak OOM
eval_indices = np.random.RandomState(42).choice(50000, size=10000, replace=False)
sample_X = tfidf_matrix_50k[eval_indices]
sample_labels = df['cluster'].iloc[eval_indices]

sil_score = silhouette_score(sample_X, sample_labels)
db_score = davies_bouldin_score(sample_X.toarray(), sample_labels)

print('\n' + '=' * 50)
print('EVALUASI CLUSTERING')
print('=' * 50)
print(f'Silhouette Score     : {sil_score:.4f} (Mendekati 1 = Klaster terpisah dengan baik)')
print(f'Davies-Bouldin Index : {db_score:.4f} (Semakin kecil = Separasi klaster semakin baik)')

Klasifikasi Sentimen pada Seluruh 50.000 Dokumen

In [ ]:
# 1. Transformasi Fitur TF-IDF & Label Target
y = df['sentiment'].map({'negative': 0, 'positive': 1})

# 2. Split Data Latih & Uji (80% latih = 40.000, 20% uji = 10.000)
X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix_50k, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Data Latih: {X_train.shape[0]:,} ulasan')
print(f'Data Uji  : {X_test.shape[0]:,} ulasan')

# 3. Model Klasifikasi (Logistic Regression)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 4. Prediksi dan Evaluasi
y_pred = clf.predict(X_test)

print('\n' + '=' * 50)
print('LAPORAN KLASIFIKASI SENTIMEN (10.000 DATA UJI)')
print('=' * 50)
print(
    classification_report(
        y_test, y_pred, target_names=['Negative (0)', 'Positive (1)']
    )
)

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
)
plt.title('Confusion Matrix Sentimen IMDB (50k Full)')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()